In [92]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(confusion_matrix, classification_report, accuracy_score, precision_score,
 recall_score, f1_score, roc_auc_score)

In [93]:
data = pd.DataFrame() 
years = ["2022"]

for y in years:
    PATH = f"../data/processed/statcast_{y}.parquet"
    data = pd.concat([data, pd.read_parquet(PATH)])

# Keep only umpire ball/strike calls
data = data[data['description'].isin(['called_strike', 'ball'])]

print(data.shape)

(380559, 121)


In [94]:
#called pitches only 
df = data[data['description'].isin(['called_strike', 'ball'])].copy()

In [95]:
x = -0.83
y = 0.83
df = df.dropna(subset=['plate_x', 'plate_z', 'sz_bot', 'sz_top'])

df['expected'] = (
    df['plate_x'].between(x,y) &
    df['plate_z'].between(df['sz_bot'], df['sz_top'])
).astype(int)

In [96]:
# if called strike 1 if ball 0 
df['outcome'] = (
    df['description'] == 'called_strike'
).astype(int)

In [97]:
# do expected & call match 
df['target'] = (
    df['outcome'] != df['expected']
).astype(int)

In [98]:
f_data = [
    'plate_x',
    'plate_z',
    'sz_top',
    'sz_bot',
    'release_speed',
    'balls',
    'strikes',
    'pitch_type',
    'stand',
    'p_throws'
]

target_col = 'target'
cat_col = ['pitch_type', 'stand','p_throws']
#categorical variables
num_col = [col for col in f_data if col not in cat_col] #numeric variables

In [99]:
X_data = df[f_data].copy()
Y_data = df[target_col].copy()

In [100]:
X_data.head()

,plate_x,plate_z,sz_top,sz_bot,release_speed,balls,strikes,pitch_type,stand,p_throws
4,0.652451,1.724418,3.720355,1.723947,94.5,1,0,FF,L,R
5,-0.299552,3.580697,3.658373,1.728693,95.6,0,0,FF,L,R
6,-1.727309,2.289997,3.541271,1.637857,94.4,3,0,FF,R,R
7,0.860265,3.49821,3.529266,1.683829,95.5,2,0,FF,R,R
13,2.652251,0.947512,3.22007,1.642547,85.9,0,1,ST,L,R


In [101]:
# numeric replace with median
for col in num_col:
    X_data[col] = X_data[col].fillna(X_data[col].median())

# categorical separate category for missing
for col in cat_col:
    X_data[col] = X_data[col].fillna('Missing')

In [102]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_cat = encoder.fit_transform(X_data[cat_col])

encoded_df = pd.DataFrame(
    encoded_cat,
    columns=encoder.get_feature_names_out(cat_col),
    index=X_data.index
)

X_data_transformed = pd.concat(
    [X_data[num_col], encoded_df],
    axis=1
)

In [103]:
X_train, X_test, y_train, y_test = train_test_split(
    X_data_transformed,
    Y_data,
    test_size=0.2,
    random_state=42,
    stratify=Y_data
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (297003, 26)
Testing set: (74251, 26)


In [104]:
lr_model = LogisticRegression(max_iter=1000,class_weight='balanced',
    random_state=42) # needed to add in balanced, we have a class imbalance

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

In [105]:
print(y_train.value_counts(normalize=True))
print(pd.Series(y_pred_lr).value_counts())

target
0    0.913233
1    0.086767
Name: proportion, dtype: float64
1    39734
0    34517
Name: count, dtype: int64


In [106]:
print("Actual:")
print(pd.Series(y_test).value_counts())

print("\nPredicted:")
print(pd.Series(y_pred_lr).value_counts())

Actual:
target
0    67809
1     6442
Name: count, dtype: int64

Predicted:
1    39734
0    34517
Name: count, dtype: int64


In [107]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[32310 35499]
 [ 2207  4235]]
              precision    recall  f1-score   support

           0       0.94      0.48      0.63     67809
           1       0.11      0.66      0.18      6442

    accuracy                           0.49     74251
   macro avg       0.52      0.57      0.41     74251
weighted avg       0.86      0.49      0.59     74251



In [108]:
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    ## add more models here 
}

results = []

for name, (pred, prob) in models.items():

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results)

print(results_df.round(4))

                 Model  Accuracy  Precision  Recall      F1  ROC-AUC
0  Logistic Regression    0.4922     0.1066  0.6574  0.1834   0.5864


In [109]:
print("Actual:")
print(pd.Series(y_test).value_counts())

print("\nPredicted:")
print(pd.Series(y_pred_lr).value_counts())

Actual:
target
0    67809
1     6442
Name: count, dtype: int64

Predicted:
1    39734
0    34517
Name: count, dtype: int64
